# 05 — Reinforcement learning

Two RL applications on top of the trained detectors:

- **Contextual-bandit decision policy (REINFORCE)** — a small policy that chooses
  `auto-human / review / auto-AI` per document to maximize a cost-sensitive utility
  (correct auto-decision +1, wrong auto-decision −3, review −0.1). Compared against
  grid-tuned fixed thresholds, always-review and no-review baselines.
- **RLCD fine-tune of TextCNN** — the same RLCD maths the Laya recipe uses (Gaussian
  noise on the answer logits, strictly proper scoring reward, REINFORCE term + soft
  cross-entropy anchor), scaled down to a from-scratch detector. Registered as
  `textcnn_rlcd` (runtime `pytorch-textcnn`), selectable per request with
  `ensembleWeight: 0` so the TextCNN family is not double-counted in the ensemble.


In [ ]:
import sys
from pathlib import Path

# Make `aitextd` importable regardless of the notebook working directory.
ROOT_HINT = Path.cwd()
while ROOT_HINT != ROOT_HINT.parent and not (ROOT_HINT / 'aitextd' / 'paths.py').exists():
    ROOT_HINT = ROOT_HINT.parent
sys.path.insert(0, str(ROOT_HINT))

from aitextd.paths import PROJECT_ROOT

DATA = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_JSON = PROJECT_ROOT / 'models.json'

RANDOM_SEED = 42
FORCE_TRAIN_RLCD = False
FORCE_TRAIN_POLICY = False


## Shared context features

The bandit's state per document: the deployed linear SVM's AI probability (sigmoid of
its margin over the TF-IDF slice) plus the 10 stylometric features — z-scored with
train-split statistics. Fast to compute for the whole canonical split.


In [ ]:
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(12)

X = joblib.load(DATA / 'tfidf_vectorizer_X.joblib')
split = joblib.load(DATA / 'split_indices.joblib')
train_idx, test_idx = split['train_idx'], split['test_idx']
y = pd.read_csv(PROJECT_ROOT / 'data' / 'processedDataset.csv', usecols=['aiGenerated'])['aiGenerated'].to_numpy()

from aitextd import paths

tfidf_dim = len(joblib.load(paths.TFIDF_VECTORIZER_JOBLIB).get_feature_names_out())
svm = joblib.load(MODELS_DIR / 'svModel.joblib')
p_svm = 1.0 / (1.0 + np.exp(-svm.decision_function(X[:, :tfidf_dim])))
style = np.asarray(X[:, tfidf_dim:].todense()) if sp.issparse(X) else np.asarray(X[:, tfidf_dim:])
C = np.column_stack([p_svm, style]).astype(np.float32)

MU, SD = C[train_idx].mean(axis=0), C[train_idx].std(axis=0) + 1e-8
CZ = (C - MU) / SD
print(f'context matrix {CZ.shape} (p_svm + {CZ.shape[1] - 1} style features)')


## Part A — contextual bandit (REINFORCE)

Utility: correct auto-decision +1, wrong auto-decision −3, review −0.1. The policy is a
softmax MLP over the context; episodes are documents, the advantage uses a running-mean
baseline. Baselines on the test split: two-stage fixed thresholds grid-tuned on train,
always-review, and no-review argmax.


In [ ]:
REWARD_AUTO_CORRECT, REWARD_AUTO_WRONG, REWARD_REVIEW = 1.0, -3.0, -0.1
EPOCHS, BATCH, LR, HIDDEN = 5, 256, 1e-3, 64


def reward_of(action, label):
    return np.where(action == 1, REWARD_REVIEW, np.where(action == label, REWARD_AUTO_CORRECT, REWARD_AUTO_WRONG))


def policy_report(action, label):
    auto = action != 1
    util = reward_of(action, label)
    return {
        'utility_mean': round(float(util.mean()), 4),
        'utility_sd': round(float(util.std()), 4),
        'review_rate': round(float((action == 1).mean()), 4),
        'auto_accuracy': round(float((action[auto] == label[auto]).mean()), 4) if auto.any() else None,
    }


best = None
for lo in np.arange(0.05, 0.95, 0.05):
    for hi in np.arange(lo + 0.05, 1.0, 0.05):
        a = np.where(C[train_idx, 0] >= hi, 2, np.where(C[train_idx, 0] >= lo, 1, 0))
        u = reward_of(a, y[train_idx]).mean()
        if best is None or u > best[0]:
            best = (u, lo, hi)
_, LO_STAR, HI_STAR = best
print(f'fixed thresholds tuned on train: review>={LO_STAR:.2f}, auto-ai>={HI_STAR:.2f}')


def fixed_policy(p):
    return np.where(p >= HI_STAR, 2, np.where(p >= LO_STAR, 1, 0))


In [ ]:
import torch.nn as nn

POLICY_FILE = MODELS_DIR / 'rlPolicyModel.pt'

if FORCE_TRAIN_POLICY:
    policy = nn.Sequential(nn.Linear(CZ.shape[1], HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, 3))
    opt = torch.optim.Adam(policy.parameters(), lr=LR)
    Ct = torch.tensor(CZ[train_idx], dtype=torch.float32)
    yt = torch.tensor(y[train_idx], dtype=torch.long)
    baseline = 0.0
    for epoch in range(EPOCHS):
        perm = torch.randperm(len(Ct))
        total = 0.0
        for start in range(0, len(perm), BATCH):
            idx = perm[start : start + BATCH]
            logits = policy(Ct[idx])
            action = torch.multinomial(torch.softmax(logits, -1), 1).squeeze(-1)
            r = torch.tensor(
                [REWARD_REVIEW if a == 1 else (REWARD_AUTO_CORRECT if a == lb else REWARD_AUTO_WRONG)
                 for a, lb in zip(action.tolist(), yt[idx].tolist())],
                dtype=torch.float32,
            )
            adv = r - baseline
            baseline = 0.95 * baseline + 0.05 * r.mean().item()
            logp = torch.log_softmax(logits, -1)[torch.arange(len(idx)), action]
            loss = -(adv * logp).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        print(f'epoch {epoch + 1}/{EPOCHS} loss={total / len(perm):.4f} baseline={baseline:.3f}')
    torch.save(
        {
            'state_dict': policy.state_dict(),
            'config': {
                'n_features': int(CZ.shape[1]),
                'hidden': HIDDEN,
                'actions': ['auto_human', 'review', 'auto_ai'],
                'reward': {'auto_correct': REWARD_AUTO_CORRECT, 'auto_wrong': REWARD_AUTO_WRONG, 'review': REWARD_REVIEW},
                'feature_mean': MU.tolist(),
                'feature_std': SD.tolist(),
                'context': ['p_svm'] + [f'style_{i}' for i in range(CZ.shape[1] - 1)],
                'fixed_thresholds': {'review': float(LO_STAR), 'auto_ai': float(HI_STAR)},
            },
        },
        POLICY_FILE,
    )
    print('saved', POLICY_FILE.name)
else:
    ckpt = torch.load(POLICY_FILE, map_location='cpu', weights_only=False)
    cfg = ckpt['config']
    policy = nn.Sequential(nn.Linear(cfg['n_features'], cfg['hidden']), nn.ReLU(), nn.Linear(cfg['hidden'], 3))
    policy.load_state_dict(ckpt['state_dict'])
    LO_STAR = cfg.get('fixed_thresholds', {}).get('review', LO_STAR)
    HI_STAR = cfg.get('fixed_thresholds', {}).get('auto_ai', HI_STAR)
    print('loaded checkpoint rlPolicyModel.pt')

policy.eval()
with torch.no_grad():
    action_probs = torch.softmax(policy(torch.tensor(CZ[test_idx], dtype=torch.float32)), -1).numpy()
rl_action = action_probs.argmax(axis=1)
bandit_results = {
    'reinforce_policy': policy_report(rl_action, y[test_idx]),
    'fixed_thresholds': policy_report(fixed_policy(C[test_idx, 0]), y[test_idx]),
    'always_review': policy_report(np.ones(len(test_idx), dtype=int), y[test_idx]),
    'no_review_argmax': policy_report((C[test_idx, 0] >= 0.5).astype(int) * 2, y[test_idx]),
}
print(pd.DataFrame(bandit_results).T.to_string())


The learned policy lands at parity with the grid-tuned fixed thresholds: with only
p_svm + style features as context, two tuned thresholds already sit near the optimum
of this utility, and the honest conclusion is that the RL machinery buys a *learned,
extensible* policy (richer context can be plugged in later), not a better number on
this feature set. The policy is saved to `models/rlPolicyModel.pt` and is **not**
registered in `models.json`: it is a decision layer over a frozen detector, not a
detector, and served behavior stays pinned.


## Part B — RLCD fine-tune of TextCNN

Starts from the supervised `textcnnModel.pt` checkpoint (same vocab, same config, so
the serving loader rebuilds it unchanged). Soft targets = 0.8·one-hot(label) +
0.2·teacher probability, clipped to [0.02, 0.98] — the same blend the System One
distillation uses. Per step: K=4 zero-mean Gaussian noise copies of the logits
(σ annealed 0.4 → 0.1), each scored by the strictly proper reward
(0.75·spherical + 1.0·ranked probability score), advantage normalized across the noise
draws; loss = REINFORCE term + soft cross-entropy anchor. CPU budget: 2 epochs over a
stratified 6,000-doc subsample of the canonical train split.


In [ ]:
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

from aitextd.architectures import TextCNNClassifier

RLCD_FILE = MODELS_DIR / 'textcnnRlcdModel.pt'
RLCD_EPOCHS, RLCD_SUBSAMPLE, RLCD_BATCH, RLCD_LR = 2, 6000, 64, 5e-5
NOISE_SIGMA, K_NOISE, W_SPH, W_RPS = (0.4, 0.1), 4, 0.75, 1.0
LABEL_WEIGHT, TEACHER_WEIGHT, TARGET_CLIP = 0.8, 0.2, (0.02, 0.98)
MAX_LEN, PAD_IDX, UNK_IDX = 200, 0, 1


def encode(tokens, vocab, max_len=MAX_LEN):
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    return ids + [PAD_IDX] * (max_len - len(ids))


def predict_probs(model, X, vocab, batch=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(torch.sigmoid(model(X[i : i + batch])).numpy())
    return np.concatenate(out)


def expected_calibration_error(y_true, probs, bins=15):
    y_true, probs = np.asarray(y_true, dtype=float), np.asarray(probs, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    edges[0], edges[-1] = -np.inf, np.inf
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (probs > lo) & (probs <= hi)
        if mask.any():
            ece += mask.sum() / len(probs) * abs(y_true[mask].mean() - probs[mask].mean())
    return round(float(ece), 4)


def evaluate(y_true, probs):
    pred = (probs >= 0.5).astype(int)
    c = {'tn': int(((pred == 0) & (y_true == 0)).sum()), 'fp': int(((pred == 1) & (y_true == 0)).sum()),
         'fn': int(((pred == 0) & (y_true == 1)).sum()), 'tp': int(((pred == 1) & (y_true == 1)).sum())}
    return {
        'accuracy': round(float(accuracy_score(y_true, pred)), 4),
        'precision': round(float(precision_score(y_true, pred)), 4),
        'recall': round(float(recall_score(y_true, pred)), 4),
        'f1': round(float(f1_score(y_true, pred)), 4),
        'roc_auc': round(float(roc_auc_score(y_true, probs)), 4),
        'ece': expected_calibration_error(y_true, probs),
        'confusion': c,
    }


In [ ]:
from sklearn.metrics import precision_score, recall_score

df = pd.read_csv(DATA / 'processedDataset.csv')
tokenized = joblib.load(DATA / 'tokenized_texts.joblib')
y_all = df['aiGenerated'].to_numpy()

sup = torch.load(MODELS_DIR / 'textcnnModel.pt', map_location='cpu', weights_only=False)
vocab, textcnn_config = sup['vocab'], sup['config']


def make_textcnn():
    return TextCNNClassifier(
        vocab_size=len(vocab), embed_dim=textcnn_config['embed_dim'],
        n_filters=textcnn_config['n_filters'], kernel_sizes=tuple(textcnn_config['kernel_sizes']),
        pad_idx=textcnn_config['pad_idx'], dropout=textcnn_config.get('dropout', 0.3),
    )


def encode_split(idx):
    return torch.tensor([encode(tokenized[i], vocab) for i in idx], dtype=torch.long)


X_test_seq = encode_split(test_idx)
y_test_all = y_all[test_idx]

if FORCE_TRAIN_RLCD:
    model = make_textcnn()
    model.load_state_dict(sup['state_dict'])
    train_sub, _ = train_test_split(train_idx, train_size=RLCD_SUBSAMPLE, random_state=RANDOM_SEED, stratify=y_all[train_idx])
    train_sub = np.asarray(train_sub)
    X_train_seq = encode_split(train_sub)

    def batch_probs(m, X, batch=256):
        m.eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(X), batch):
                out.append(torch.sigmoid(m(X[i : i + batch])).numpy())
        return np.concatenate(out)

    t_ai = np.clip(LABEL_WEIGHT * y_all[train_sub] + TEACHER_WEIGHT * batch_probs(model, X_train_seq), *TARGET_CLIP)
    targets = torch.tensor(np.stack([1.0 - t_ai, t_ai], axis=1), dtype=torch.float32)

    optimizer = torch.optim.AdamW(model.parameters(), lr=RLCD_LR)
    for epoch in range(RLCD_EPOCHS):
        sigma = NOISE_SIGMA[0] + (NOISE_SIGMA[1] - NOISE_SIGMA[0]) * epoch / max(1, RLCD_EPOCHS - 1)
        model.train()
        order = np.random.default_rng(RANDOM_SEED + epoch).permutation(len(train_sub))
        total = 0.0
        for start in range(0, len(order), RLCD_BATCH):
            idx = order[start : start + RLCD_BATCH]
            xb, tb = X_train_seq[idx], targets[idx]
            logits = model(xb)
            clean2 = torch.stack([-logits, logits], dim=-1)
            eps = torch.randn((K_NOISE, *clean2.shape)) * sigma
            noisy = clean2.detach().unsqueeze(0) + eps
            probs = torch.softmax(noisy, dim=-1)
            sph = (probs * tb.unsqueeze(0)).sum(-1) / probs.pow(2).sum(-1).sqrt()
            rps = -0.5 * ((torch.cumsum(probs, -1) - torch.cumsum(tb, -1).unsqueeze(0)) ** 2).sum(-1)
            reward = W_SPH * sph + W_RPS * rps
            adv = (reward - reward.mean(0, keepdim=True)) / (reward.std() + 1e-6)
            logp = -(((noisy - clean2.unsqueeze(0)) ** 2).sum(-1)) / (2 * sigma**2)
            loss = -(adv * logp).mean() + F.binary_cross_entropy_with_logits(logits, tb[:, 1])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        print(f'epoch {epoch + 1}/{RLCD_EPOCHS} loss={total / len(train_sub):.4f} sigma={sigma:.2f}')
    torch.save({'state_dict': model.state_dict(), 'vocab': vocab, 'config': textcnn_config}, RLCD_FILE)
    print('saved', RLCD_FILE.name)
else:
    model = make_textcnn()
    model.load_state_dict(torch.load(RLCD_FILE, map_location='cpu', weights_only=False)['state_dict'])
    print('loaded checkpoint textcnnRlcdModel.pt')

supervised_model = make_textcnn()
supervised_model.load_state_dict(sup['state_dict'])

supervised_metrics = evaluate(y_test_all, predict_probs(supervised_model, X_test_seq, vocab))
rlcd_metrics = evaluate(y_test_all, predict_probs(model, X_test_seq, vocab))
rlcd_results = {'textcnn (supervised)': supervised_metrics, 'textcnn_rlcd': rlcd_metrics}
for mid, m in rlcd_results.items():
    c = m['confusion']
    print(f"{mid:22s} acc={m['accuracy']:.4f} f1={m['f1']:.4f} roc_auc={m['roc_auc']:.4f} "
          f"ece={m['ece']:.4f} tn={c['tn']} fp={c['fp']} fn={c['fn']} tp={c['tp']}")


On the canonical test split the RLCD fine-tune lands at parity with its supervised
starting point (accuracy 0.9900 vs 0.9905, ROC-AUC 0.9994 both) — the RL objective
reproduces supervised quality here rather than beating it; with a near-ceiling teacher
there is little headroom for the reward to find signal beyond the labels. The value of
the recipe is that it trains a detector *without hard labels* when soft targets exist.


## Serving check and registration

The RLCD checkpoint reuses the existing `pytorch-textcnn` runtime verbatim — same
config keys, same vocabulary — so no serving code changes. The cell below loads it
through the exact serving path and registers it in `models.json` (only with
`FORCE_TRAIN_RLCD`).


In [ ]:
from aitextd import loaders, predictors
from aitextd.text import preprocess_persian_text

bundle = loaders.LOADERS['pytorch-textcnn']({'filename': 'textcnnRlcdModel.pt', 'runtime': 'pytorch-textcnn'})
sample = 'هوش مصنوعی یکی از مهم‌ترین فناوری‌های قرن بیست‌ویکم است و تأثیر عمیقی بر اقتصاد گذاشته است.'
cleaned = preprocess_persian_text(sample)
print('serving-path prediction:', round(predictors.PREDICTORS['pytorch-textcnn'](bundle, sample, cleaned, {}), 4))


def metrics_json(m):
    return {'accuracy': m['accuracy'], 'precision': m['precision'], 'recall': m['recall'],
            'f1Score': m['f1'], 'rocAuc': m['roc_auc']}


def confusion_json(m):
    c = m['confusion']
    return {'trueNegative': c['tn'], 'falsePositive': c['fp'], 'falseNegative': c['fn'], 'truePositive': c['tp']}


if FORCE_TRAIN_RLCD:
    entry = {
        'id': 'textcnn_rlcd',
        'name': 'TextCNN (RLCD fine-tune)',
        'nameFa': 'شبکه کانولوشنی متن (فاین‌تیون RLCD)',
        'description': (
            'The deployed TextCNN classifier fine-tuned with RLCD (reinforcement learning with '
            'contrastive decoding): Gaussian noise is drawn on the answer logits, a strictly proper '
            'scoring reward (spherical + ranked probability score) scores each noisy candidate against '
            'soft targets distilled from the supervised model (0.8 ground-truth / 0.2 teacher blend), '
            'and a REINFORCE term plus a soft cross-entropy anchor update the weights. An RL experiment '
            'selectable per request; ensembleWeight 0 keeps it out of the ensemble vote so the TextCNN '
            'family is not double-counted.'
        ),
        'descriptionFa': (
            'همان دسته‌بند TextCNN مستقر که با RLCD (یادگیری تقویتی با رمزگشایی متضاد) فاین‌تیون شده است: '
            'نویز گاوسی روی لاجیت پاسخ‌ها کشیده می‌شود، یک پاداش امتیازدهیِ کاملاً درست (کروی + امتیاز احتمال '
            'رتبه‌ای) هر نامزد نویزی را در برابر هدف‌های نرم تقطیرشده از مدل نظارت‌شده (ترکیب ۰٫۸ برچسب واقعی و '
            '۰٫۲ دبیر) می‌سنجد و یک جملهٔ REINFORCE به‌همراه لنگرِ آنتروپی متقاطع نرم وزن‌ها را به‌روزرسانی می‌کند. '
            'یک آزمایش یادگیری تقویتی که به‌ازای هر درخواست قابل انتخاب است؛ وزن صفرِ رأی ترکیبی آن را از رأی‌گیری '
            'کنار گذاشته تا خانوادهٔ TextCNN دوبار شمرده نشود.'
        ),
        'algorithm': 'TextCNN + RLCD fine-tune',
        'type': 'dl',
        'version': '1.0',
        'filename': 'textcnnRlcdModel.pt',
        'runtime': 'pytorch-textcnn',
        'ensembleWeight': 0.0,
        'metrics': metrics_json(rlcd_metrics),
        'confusionMatrix': confusion_json(rlcd_metrics),
        'training': {
            'totalSamples': len(train_idx) + len(test_idx),
            'trainSamples': RLCD_SUBSAMPLE,
            'testSamples': len(test_idx),
            'featureCount': len(vocab),
            'testSplitRatio': 0.2,
            'randomSeed': RANDOM_SEED,
            'dataset': 'persian_ai_generated_final.csv + blogs.csv (RLCD fine-tune of textcnnModel.pt, 6000-doc stratified subsample)',
        },
    }
    registry = json.loads(MODELS_JSON.read_text(encoding='utf-8'))
    registry = [m for m in registry if m['id'] != 'textcnn_rlcd']
    after = next(i for i, m in enumerate(registry) if m['id'] == 'textcnn') + 1
    registry.insert(after, entry)
    MODELS_JSON.write_text(json.dumps(registry, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('registered textcnn_rlcd')
else:
    print('skipped (FORCE_TRAIN_RLCD = False)')


## Summary


In [ ]:
print('decision policies (test-split utility):')
print(pd.DataFrame(bandit_results).T.to_string())
print()
print('detectors (canonical test split):')
print(pd.DataFrame({mid: {k: m[k] for k in ('accuracy', 'f1', 'roc_auc', 'ece')} for mid, m in rlcd_results.items()}).T.to_string())
